# YOLO11n — области анонимизации, вариант 1

Первый локальный эксперимент: `anonymized_region`, без OCR и восстановления скрытого содержимого.

**Сначала вручную проверьте новые 240 кропов.** Старые 126 кадров и `masks-7.json` относятся только к отдельному аудиту. Они не обучающая выборка.

По умолчанию обучение и итоговый holdout выключены. Run All сейчас безопасен: он не примет непроверенные подсказки за разметку.

In [8]:
from pathlib import Path
import sys, json, os

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'backend/core.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from training.mask_finetune import EXPERIMENT, load_json, export_reviewed
from training.yolo_masks import runtime, device_name, train_masks, evaluate_masks, TRAIN_ARGS

ANNOTATIONS = EXPERIMENT / 'annotation/reviewed_masks.json'  # сюда сохраните новый JSON из разметчика
RUN_NAME = 'pilot_01'
RUN_TRAINING = True  # включить только после проверки новых 240 кропов
RESUME = False  # True только после прерванного, НЕ завершённого запуска
RUN_HOLDOUT = False  # один раз после фиксации модели и порогов на val
print('Experiment:', EXPERIMENT)
print('Annotations:', ANNOTATIONS)

Experiment: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/YOLO11/variant_01_anonymized_regions
Annotations: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/YOLO11/variant_01_anonymized_regions/annotation/reviewed_masks.json


## 1. Окружение

Используйте ядро проекта `.venv/bin/python`. На текущем компьютере зависимости уже подготовлены отдельно в `artifacts/yolo11_runtime`; базовые PyTorch/torchvision не обновлялись.

Ячейка установки нужна только при переносе/восстановлении. Используется установленный `uv`, без переустановки основной ML-среды. Ни фотографии, ни результаты в Hub не отправляются.

In [9]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    import subprocess, shutil
    uv = shutil.which('uv')
    if not uv:
        raise RuntimeError('Нужен uv для изолированной установки; основное окружение не менять.')
    subprocess.check_call([uv, 'pip', 'install', '--python', sys.executable,
        '--target', str(ROOT / 'artifacts/yolo11_runtime'), '--no-deps',
        '--cache-dir', str(ROOT / '.uv-cache'), '-r', str(EXPERIMENT / 'requirements-runtime.txt')])
YOLO = runtime()
DEVICE = device_name()
print('Device:', DEVICE)
print('CPU означает, что MPS/CUDA недоступны текущему ядру; запуск будет медленнее.')

Device: mps
CPU означает, что MPS/CUDA недоступны текущему ядру; запуск будет медленнее.


## 2. Проверка разметки и экспорт

В разметчике нажмите «Проверено» для каждого кадра, в том числе без размытия. Скачайте JSON и задайте его путь в `ANNOTATIONS`.

Экспорт останавливается при неверном наборе, непроверенном кадре, выходе bbox за кроп или изменении исходных изображений. Только подтверждённый пустой список становится отрицательным примером. Train/val/holdout разделены по машинам и одинаковым кадрам. В `data.yaml` нет контрольной части.

In [10]:
READY = False
if ANNOTATIONS.is_file():
    manifest = export_reviewed(EXPERIMENT / 'annotation/mask_plan.json', ANNOTATIONS, EXPERIMENT / 'data')
    READY = True
    print(json.dumps(manifest['splits'], indent=2, ensure_ascii=False))
    if any(v['negative_images'] == 0 for v in manifest['splits'].values()):
        print('ВНИМАНИЕ: есть части без отрицательных кадров. Не удаляйте настоящие маски ради баланса; специфичность на таких частях не измерить.')
else:
    print('Разметка ещё не загружена. Откройте annotation/annotate_masks.html; после проверки укажите новый JSON.')

{
  "train": {
    "images": 160,
    "regions": 191,
    "negative_images": 0
  },
  "val": {
    "images": 40,
    "regions": 49,
    "negative_images": 0
  },
  "holdout": {
    "images": 40,
    "regions": 45,
    "negative_images": 0
  }
}
ВНИМАНИЕ: есть части без отрицательных кадров. Не удаляйте настоящие маски ради баланса; специфичность на таких частях не измерить.


## 3. Настройки обучения

Старт — официальный YOLO11n COCO `.pt`, не ONNX детектора номеров из прошлого сравнения. Новая задача — один класс `anonymized_region`.

50 эпох максимум, patience 10, AdamW, LR 0.001, batch 8, 640 px. Mosaic/MixUp выключены, умеренные геометрические и цветовые изменения. Сохраняются `best.pt` и `last.pt`, а не все эпохи. Best выбирает Ultralytics по validation fitness; дополнительно измеряем полноту маскирования. Эти настройки — исходный пилот, не гарантированный оптимум.

In [11]:
print(json.dumps(TRAIN_ARGS, indent=2))
provenance = load_json(EXPERIMENT / 'weights/pretrained/model.json')
print('Source:', provenance['url'])
print('SHA256:', provenance['sha256'])
RUN_DIR = EXPERIMENT / 'runs' / RUN_NAME

{
  "epochs": 50,
  "patience": 10,
  "imgsz": 640,
  "batch": 8,
  "workers": 0,
  "optimizer": "AdamW",
  "lr0": 0.001,
  "lrf": 0.05,
  "weight_decay": 0.0005,
  "cos_lr": true,
  "warmup_epochs": 3,
  "seed": 20260920,
  "deterministic": true,
  "amp": false,
  "cache": false,
  "mosaic": 0.0,
  "mixup": 0.0,
  "copy_paste": 0.0,
  "degrees": 0.0,
  "translate": 0.05,
  "scale": 0.15,
  "shear": 0.0,
  "perspective": 0.0,
  "flipud": 0.0,
  "fliplr": 0.5,
  "hsv_h": 0.015,
  "hsv_s": 0.3,
  "hsv_v": 0.15,
  "save": true,
  "save_period": -1,
  "plots": true
}
Source: https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt
SHA256: 0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1


## 4. Обучение — явный запуск

После проверки данных измените `RUN_TRAINING=True` в первой ячейке и выполните ноутбук. На Apple Silicon автоматически выбирается MPS, если доступен. Прогресс показывает эпоху, её длительность, оставшиеся эпохи и ETA до максимума; early stopping может закончить раньше.

При прерывании не удаляйте JSON/веса: выставьте `RESUME=True` с прежними разметкой, настройками и RUN_NAME. После завершённого эксперимента для нового запуска используйте новое имя. Если `last.pt` ещё не создан (не закончилась первая эпоха), сохраните старую папку и выберите другое RUN_NAME.

In [12]:
if RUN_TRAINING:
    if not READY:
        raise RuntimeError('Сначала загрузите и проверьте ручную разметку.')
    RUN_DIR = train_masks(ANNOTATIONS, run_name=RUN_NAME, resume=RESUME, device=DEVICE)
else:
    print('Обучение выключено. Никакие веса не изменены.')

Device: mps; train/val only, holdout not used. Output: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/YOLO11/variant_01_anonymized_regions/runs/pilot_01
Ultralytics 8.3.241 🚀 Python-3.11.9 torch-2.14.0 MPS (Apple M4 Pro)
engine/trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/YOLO11/variant_01_anonymized_regions/data/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.3, hsv_v=0.15, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, lin

/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:1072: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      2.25G      1.893      3.375      1.914         10        640: 100% ━━━━━━━━━━━━ 20/20 2.1it/s 9.4s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.1it/s 1.4s1.0s
                   all         40         49    0.00275      0.673    0.00321   0.000811
Эпоха 1/50: 0.19 мин · осталось до 49 эпох · ETA до лимита 9.2 мин (early stopping может закончить раньше)

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50       2.2G      1.466      2.313      1.539         10        640: 0% ──────────── 0/20  0.4s

/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:1072: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50      3.22G       1.63      1.878      1.667         10        640: 100% ━━━━━━━━━━━━ 20/20 3.6it/s 5.6s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 3.7it/s 0.8s0.6s
                   all         40         49    0.00292      0.714     0.0574     0.0222
Эпоха 2/50: 0.11 мин · осталось до 48 эпох · ETA до лимита 7.2 мин (early stopping может закончить раньше)

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      2.21G      1.525      1.597      1.516         11        640: 0% ──────────── 0/20  0.4s

...

      24/50      3.23G     0.9609     0.5946       1.14          9        640: 100% ━━━━━━━━━━━━ 20/20 3.7it/s 5.4s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 3.1it/s 1.0s0.7s
                   all         40         49      0.935      0.874      0.898      0.509
Эпоха 24/50: 0.11 мин · осталось до 26 эпох · ETA до лимита 3.0 мин (early stopping может закончить раньше)

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      25/50      2.22G      1.097     0.6838      1.136         11        640: 0% ──────────── 0/20  0.5s

/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:1072: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      25/50      3.23G     0.9524     0.5908      1.134          9        640: 100% ━━━━━━━━━━━━ 20/20 3.6it/s 5.5s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 3.5it/s 0.8s0.6s
                   all         40         49      0.952      0.857       0.91      0.493
Эпоха 25/50: 0.11 мин · осталось до 25 эпох · ETA до лимита 2.9 мин (early stopping может закончить раньше)

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      26/50       3.2G     0.7699      0.546      1.132          8        640: 0% ──────────── 0/20  0.5s

/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:1072: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      26/50      3.23G     0.9125     0.5678      1.142         11        640: 100% ━━━━━━━━━━━━ 20/20 3.7it/s 5.4s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 3.5it/s 0.9s0.6s
                   all         40         49      0.972      0.857       0.92      0.498
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 16, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.
Эпоха 26/50: 0.11 мин · осталось до 24 эпох · ETA до лимита 2.7 мин (early stopping может закончить раньше)

26 epochs completed in 0.050 hours.
Optimizer stripped from /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/YOLO11/variant_01_anonymized_regions/runs/pilot_01/weights/last.pt, 5.5MB
Optimizer stripped from /Users/elvsevolod/Desktop/у

## 5. Validation и качество масок

После обучения автоматически считаются bbox mAP и покрытие ручных масок на 40 validation-кропах. Основные дополнительные показатели: доля площади эталона, доля областей с покрытием ≥90%, площадь вне эталона, ложные маски на отрицательных кадрах.

Порог 0.25 и запас 0 — начальная точка. Другие значения проверяем только на `val`, каждый результат — в новом каталоге. Здесь нет автоматического подбора на holdout и нет замены модели в MVP.

In [13]:
CONFIDENCE = 0.25
MARGIN = 0.0  # доля ширины/высоты рамки, добавляемая с каждой стороны
if RUN_TRAINING:
    validation = evaluate_masks(RUN_DIR / 'weights/best.pt', ANNOTATIONS,
        RUN_DIR / 'mask_validation_025_margin0', split='val', confidence=CONFIDENCE,
        margin=MARGIN, device=DEVICE)
    print(json.dumps(validation, indent=2, ensure_ascii=False))
else:
    print('Validation будет рассчитана после явно запущенного обучения.')

Ultralytics 8.3.241 🚀 Python-3.11.9 torch-2.14.0 MPS (Apple M4 Pro)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6917.9±1392.0 MB/s, size: 305.2 KB)
val: Scanning /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/YOLO11/variant_01_anonymized_regions/data/labels/val.cache... 40 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 40/40 216.5Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 4.6it/s 1.1s0.3s
                   all         40         49      0.875      0.859      0.874      0.523
Speed: 0.2ms preprocess, 15.8ms inference, 0.0ms loss, 1.1ms postprocess per image
{
  "split": "val",
  "weights_sha256": "7309199f0dbda8c23bf6306bb20580e9d3f03c0710a9071da53c949960e89f95",
  "annotations_sha256": "86a88b9e04d8ae5a126d6d4341c384824b9cbf876a00438977af0a1837d4170a",
  "confidence": 0.25,
  "margin_fraction_each_side": 0.0,
  "iou_nms": 0.45,
  "coverage": {
    "refe

## 6. Отдельный holdout — не для подбора

Оставьте выключенным, пока не выбраны модель, confidence и margin по validation. После фиксации выставьте `RUN_HOLDOUT=True` и выполните только эту ячейку. Не используйте полученный результат для повторного подбора, называя его независимым.

Даже хорошее покрытие ещё не доказывает пользу для ReID: следующим отдельным этапом сравним один и тот же ReID с масками и без них. MVP этот ноутбук не меняет.

In [14]:
if RUN_HOLDOUT:
    if not READY:
        raise RuntimeError('Нет проверенной разметки.')
    holdout = evaluate_masks(RUN_DIR / 'weights/best.pt', ANNOTATIONS,
        RUN_DIR / 'mask_holdout_final', split='holdout', confidence=CONFIDENCE,
        margin=MARGIN, device=DEVICE)
    print(json.dumps(holdout, indent=2, ensure_ascii=False))
else:
    print('Независимая контрольная проверка выключена.')

Независимая контрольная проверка выключена.


## Что сохранить

`runs/<RUN_NAME>/experiment.json`, `results.csv`, `epoch_times.json`, `weights/best.pt`, `weights/last.pt`, `mask_validation_*/metrics.json`. Не удаляйте исходный проверенный JSON.

Источники API: [обучение/MPS/resume](https://docs.ultralytics.com/modes/train/), [формат YOLO](https://docs.ultralytics.com/datasets/detect/), [callbacks](https://docs.ultralytics.com/usage/callbacks/). Подробности протокола и лицензий — `README.md`.